In [ ]:
demo_order = {
    "order_id": 10001,
    "customer": {
        "user_id": 1,
        "email": "alice@example.com",
    },
    "items": [
        {
            "sku": "ESS-001",
            "quantity": 2,
            "price": "129.99",
        }
    ],
}

In [2]:
import json

In [3]:
# json.dumps()

json_string = json.dumps(
    demo_order,
    ensure_ascii=False,
    indent=2
)

In [4]:
print(json_string)

{
  "order_id": 10001,
  "customer": {
    "user_id": 1,
    "email": "alice@example.com"
  },
  "items": [
    {
      "sku": "ESS-001",
      "quantity": 2,
      "price": "129.99"
    }
  ]
}


In [5]:
print(type(json_string))

<class 'str'>


In [6]:
#json.loads()
parsed_order = json.loads(json_string)

In [7]:
parsed_order

{'order_id': 10001,
 'customer': {'user_id': 1, 'email': 'alice@example.com'},
 'items': [{'sku': 'ESS-001', 'quantity': 2, 'price': '129.99'}]}

In [8]:
parsed_order["customer"]["email"]

'alice@example.com'

In [14]:
#json.dump() 
from pathlib import Path
with (Path("data/demo_order.json")).open("w", encoding = "utf-8") as f:
    json.dump(
        demo_order,
        f,
        ensure_ascii=False,
        indent=2,
    )

In [17]:
#json.load()

with (Path("data/demo_order.json")).open("r", encoding = "utf-8") as f:
    loaded_order = json.load(f)

In [18]:
loaded_order

{'order_id': 10001,
 'customer': {'user_id': 1, 'email': 'alice@example.com'},
 'items': [{'sku': 'ESS-001', 'quantity': 2, 'price': '129.99'}]}

In [19]:
import csv

In [20]:
# csv.DictWriter

In [25]:
# csv.DictReader
with (Path("data/customers.csv")).open("r", encoding = "utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

In [24]:
print(reader)

In [27]:
rows[:1]

[{'user_id': '1',
  'email': 'alice@example.com',
  'country': 'US',
  'segment': 'Premium',
  'status': 'ACTIVE'}]

In [28]:
clean_customer_rows = []

In [31]:
with (Path("data/customers.csv")).open("r", encoding = "utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        clean_customer_rows.append(
            {
                "user_id": int(row["user_id"]),
                "email": row["email"],
                "country": row["country"],
                "segment": row["segment"].lower(),
                "status": row["status"].lower()
            }
        )

In [32]:
clean_customer_rows

[{'user_id': 1,
  'email': 'alice@example.com',
  'country': 'US',
  'segment': 'premium',
  'status': 'active'},
 {'user_id': 2,
  'email': 'bob@example.com',
  'country': 'US',
  'segment': 'new',
  'status': 'active'},
 {'user_id': 3,
  'email': 'carlos@example.com',
  'country': 'MX',
  'segment': 'regular',
  'status': 'active'}]

In [34]:
import pandas as pd

In [35]:
#pd.read_csv
customer_df = pd.read_csv(Path("data/customers.csv"))

In [36]:
customer_df

,user_id,email,country,segment,status
0,1,alice@example.com,US,Premium,ACTIVE
1,2,bob@example.com,US,New,active
2,3,carlos@example.com,MX,Regular,ACTIVE


In [37]:
customer_df.head(1)

,user_id,email,country,segment,status
0,1,alice@example.com,US,Premium,ACTIVE


In [38]:
customer_df.columns

Index(['user_id', 'email', 'country', 'segment', 'status'], dtype='str')

In [39]:
customer_df.dtypes

user_id    int64
email        str
country      str
segment      str
status       str
dtype: object

In [40]:
customer_df[["user_id", "email", "status"]]

,user_id,email,status
0,1,alice@example.com,ACTIVE
1,2,bob@example.com,active
2,3,carlos@example.com,ACTIVE


In [41]:
# df.assign
customers_normalized_df= customer_df.assign(
    segment=customer_df["segment"].str.lower(),
    status=customer_df["status"].str.lower(),
)

In [42]:
customers_normalized_df

,user_id,email,country,segment,status
0,1,alice@example.com,US,premium,active
1,2,bob@example.com,US,new,active
2,3,carlos@example.com,MX,regular,active


In [45]:
# df.query
df1 = customers_normalized_df.query("segment == 'premium'")

In [46]:
df1

,user_id,email,country,segment,status
0,1,alice@example.com,US,premium,active


In [48]:
# df.groupby
df2 = customers_normalized_df.groupby("country").agg(customers_count = ("user_id", "count"))

In [49]:
df2

,customers_count
country,
MX,1
US,2


In [51]:
# df.itertuples
rows_for_db = []
for row in customers_normalized_df.itertuples(index=False):
    rows_for_db.append(
        (
        int(row.user_id),
        row.email,
        row.country,
        row.segment,
        row.status
        )
    )

In [52]:
rows_for_db

[(1, 'alice@example.com', 'US', 'premium', 'active'),
 (2, 'bob@example.com', 'US', 'new', 'active'),
 (3, 'carlos@example.com', 'MX', 'regular', 'active')]

In [55]:
pip install psycopg

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# psycopg.connect
# psycopg.execute
with psycopg.connect(host, user) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1")
        result = cur.fetchone
    conn.commit()

In [57]:
import os                                                                                                                                                                                                                                                                                                                                                                                                                         
PG_DSN = (                                                                                                                                                                                                             
 f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"                                                                                                    
 f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}"                                                                                                                                                            
 f"/{os.environ['POSTGRES_DB']}"                                                                                                                                                                            
)       

In [70]:
# psycopg.connect
# psycopg.execute
import psycopg2
with psycopg2.connect(PG_DSN) as conn:
     with conn.cursor() as cur:
        cur.execute("SELECT 1")
        result = cur.fetchone()
result

(1,)

In [71]:
with psycopg2.connect(PG_DSN) as conn:
     with conn.cursor() as cur:
         cur.execute(
             """
             INSER INTO public.customer(...)
             VALUES (%s, %s, %s)
             """
             ,
             row_for_db
         )
        conn.commit()

SyntaxError: incomplete input (493559432.py, line 4)

In [ ]:
with psycopg2.connect(PG_DSN) as conn:
     with conn.cursor() as cur:
         cur.executemanu(
             """
             INSER INTO public.customer(...)
             VALUES (%s, %s, %s)
             """
             ,
             row_for_db
         )
        conn.commit()

In [75]:
with (Path("data/orders.json")).open("r", encoding = "utf-8") as f:
    orders_data = json.load(f)

In [76]:
orders_data[0]

{'order_id': 10001,
 'cart_id': 1,
 'customer': {'user_id': 1, 'city': 'Chicago'},
 'status': 'CONFIRMED',
 'created_at': '2026-05-01T10:15:00Z',
 'channel': {'name': 'mobile_app',
  'utm': {'source': 'push', 'campaign': 'spring_sale'}},
 'payment': {'payment_id': 'pay_10001',
  'status': 'PAID',
  'amount': '352.38',
  'currency': 'USD',
  'attempts': [{'provider': 'stripe',
    'status': 'failed',
    'attempted_at': '2026-05-01T10:15:45Z',
    'error': {'code': 'insufficient_funds'}},
   {'provider': 'stripe',
    'status': 'success',
    'attempted_at': '2026-05-01T10:16:12Z',
    'error': None}]},
 'items': [{'line_no': 1,
   'product_id': 1,
   'sku': 'ESS-001',
   'quantity': '2',
   'unit_price': '129.99',
   'category_path': ['beauty', 'makeup'],
   'adjustments': [{'type': 'discount',
     'amount': '-10.00',
     'reason': 'campaign'},
    {'type': 'tax', 'amount': '12.00', 'reason': 'sales_tax'}]},
  {'line_no': 2,
   'product_id': 2,
   'sku': 'EYE-002',
   'quantity': '1'

In [81]:
order_rows = []
for order in orders_data:
    order_rows.append(
            {
                "order_id": order["order_id"],
                "user_id": order["customer"]["user_id"],
                "city": order["customer"]["city"],
                "created_at": order["created_at"],
                "status": order["status"].lower()
            }
    )

In [82]:
orders_df = pd.DataFrame(order_rows)

In [83]:
orders_df

,order_id,user_id,city,created_at,status
0,10001,1,Chicago,2026-05-01T10:15:00Z,confirmed
1,10002,2,Houston,2026-05-02T12:00:00Z,confirmed


In [87]:
rows_to_insert = []

for row in orders_df.itertuples(index=False):
    rows_to_insert.append(
        (
            row.order_id,
            row.user_id,
            row.city,
            row.created_at,
            row.status
        )
    )

In [88]:
rows_to_insert

[(10001, 1, 'Chicago', '2026-05-01T10:15:00Z', 'confirmed'),
 (10002, 2, 'Houston', '2026-05-02T12:00:00Z', 'confirmed')]

In [89]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE SCHEMA stg IF NOT EXISTS;
            CREATE TABLE stg.orders(
                order_id BIGINT PRIMARY KEY,
                user_id BIGINT,
                city TEXT,
                created_at TIMESTAMPTZ,
                status TEXT
            )
            """
        )

In [90]:
conn.commit()

In [93]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO stg.orders(
                order_id,
                user_id,
                city,
                created_at,
                status
            )
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (order_id)
            DO UPDATE SET
                user_id = EXCLUDED.user_id,
                city = EXCLUDED.city,
                created_at = EXCLUDED.created_at,
                status = EXCLUDED.status
            """,
            rows_to_insert,
        )

In [94]:
conn.commit()

In [95]:
#requests
import requests

In [96]:
response = requests.get(
    "https://dummyjson.com/products",
    params={
        "limit":3,
        "skip":0
    },
    timeout = 10,
)

In [97]:
response.status_code

200

In [99]:
response.raise_for_status()

In [100]:
payload = response.json()

In [101]:
payload.keys()

dict_keys(['products', 'total', 'skip', 'limit'])

In [102]:
payload["products"][0]

{'id': 1,
 'title': 'Essence Mascara Lash Princess',
 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.',
 'category': 'beauty',
 'price': 9.99,
 'discountPercentage': 10.48,
 'rating': 2.56,
 'stock': 99,
 'tags': ['beauty', 'mascara'],
 'brand': 'Essence',
 'sku': 'BEA-ESS-ESS-001',
 'weight': 4,
 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99},
 'warrantyInformation': '1 week warranty',
 'shippingInformation': 'Ships in 3-5 business days',
 'availabilityStatus': 'In Stock',
 'reviews': [{'rating': 3,
   'comment': 'Would not recommend!',
   'date': '2025-04-30T09:41:02.053Z',
   'reviewerName': 'Eleanor Collins',
   'reviewerEmail': 'eleanor.collins@x.dummyjson.com'},
  {'rating': 4,
   'comment': 'Very satisfied!',
   'date': '2025-04-30T09:41:02.053Z',
   'reviewerName': 'Lucas Gordon',
   'reviewerEmail': 'lucas.gordon@x.d

In [103]:
df = pd.DataFrame(
    [
        {
            "product_id": product["id"],
            "price": product["price"],
            "stock": product["stock"],
        }
        for product in payload["products"]
    ]
)

In [104]:
df

,product_id,price,stock
0,1,9.99,99
1,2,19.99,34
2,3,14.99,89


In [3]:
import requests
products = []
limit = 30
skip = 0
max_rows = 60
while len(products) < max_rows:
    response = requests.get(
            "https://dummyjson.com/products",
                params={
                    "limit":limit,
                    "skip":skip,
                },
                timeout = 10,
    )
    response.raise_for_status()
    page = response.json()
    products.extend(page["products"])
    skip +=limit
    if skip >= page["total"]:
        break

In [5]:
products = products[:max_rows]

In [6]:
products

[{'id': 1,
  'title': 'Essence Mascara Lash Princess',
  'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.',
  'category': 'beauty',
  'price': 9.99,
  'discountPercentage': 10.48,
  'rating': 2.56,
  'stock': 99,
  'tags': ['beauty', 'mascara'],
  'brand': 'Essence',
  'sku': 'BEA-ESS-ESS-001',
  'weight': 4,
  'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99},
  'warrantyInformation': '1 week warranty',
  'shippingInformation': 'Ships in 3-5 business days',
  'availabilityStatus': 'In Stock',
  'reviews': [{'rating': 3,
    'comment': 'Would not recommend!',
    'date': '2025-04-30T09:41:02.053Z',
    'reviewerName': 'Eleanor Collins',
    'reviewerEmail': 'eleanor.collins@x.dummyjson.com'},
   {'rating': 4,
    'comment': 'Very satisfied!',
    'date': '2025-04-30T09:41:02.053Z',
    'reviewerName': 'Lucas Gordon',
    'reviewe

In [7]:
len(products)

60